# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Convert metadata to dictionary for inspection
metadata = dataset.metadata.to_json()
print(f"{metadata['name']} (version {metadata['version']}):\n{metadata['description']}")

## 2. Data Overview
Review the available record sets, their fields, and their `@id`s.

The main protein data in this dataset is stored in a record set, which we will enumerate and inspect.

In [ ]:
# List the available record sets and their IDs in the dataset
print('Available record sets and their @id:')
for rec in dataset.record_sets():
    print(f"- {rec['@id']}: {rec.get('name', '')}")

# For demonstration, inspect the fields/columns of a main record set
# Let's select the first record set (likely the main protein data)
record_sets = list(dataset.record_sets())
if record_sets:
    protein_record_set_id = record_sets[0]['@id']
    print(f"\nFields for record set {protein_record_set_id}:")
    fields = record_sets[0]['field'] if 'field' in record_sets[0] else []
    for f in fields:
        if isinstance(f, dict):
            print(f"- {f['@id']}: {f.get('name','')}")
        else:
            print(f"- {f}")
else:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We use the record set and field `@id`s found above.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# We'll collect all available record set @id's
record_set_ids = [rec['@id'] for rec in dataset.record_sets()]

dataframes = {}
for rec_id in record_set_ids:
    print(f"Loading records from record set: {rec_id}")
    try:
        df = pd.DataFrame(dataset.records(record_set=rec_id))
        print(f"Loaded {len(df)} records for {rec_id}.")
        dataframes[rec_id] = df
    except Exception as e:
        print(f"  Could not load {rec_id}: {e}")

# Show the main columns for main protein DataFrame
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nMain DataFrame columns for {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()
else:
    print('No dataframes could be loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply basic data preparation and EDA, such as filtering by numeric values and normalizing fields.

We'll:
- Select a numeric field (for example, '`mw`' or '`coverage_percent`')
- Filter by a threshold (e.g., high MW proteins)
- Normalize this field
- Group by a category (if available, e.g., '`sample`' or similar)

In [ ]:
# For demonstration, inspect the first loaded DataFrame
main_rs_id = list(dataframes.keys())[0]
df = dataframes[main_rs_id]

# Try to auto-detect a numeric field: prioritize ['mw', 'coverage_percent', 'peptide_count', 'abundance']
preferred_fields = ['mw', 'molecular_weight', 'MW', 'peptide_count', 'coverage_percent', 'abundance', 'norm_abundance']
numeric_field = None
for f in preferred_fields:
    if f in df.columns:
        numeric_field = f
        break
if numeric_field is None:
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field = c
            break

if numeric_field is not None:
    print(f"Using field '{numeric_field}' for numeric analysis.")
    # Remove rows with missing data for this field
    df = df[pd.to_numeric(df[numeric_field], errors='coerce').notnull()]
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].quantile(0.75)  # top quartile
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where '{numeric_field}' > {threshold:.2f} (top quartile):")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}':")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if available
    # Try to auto-detect group/categorical field
    group_candidates = ['sample', 'modification', 'accession', 'protein_id']
    group_field = None
    for g in group_candidates:
        if g in df.columns and df[g].nunique() > 1 and df[g].nunique() < 20:
            group_field = g
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame().reset_index()
        print(f"\nGrouped by {group_field} (mean {numeric_field}):")
        print(grouped_df)
    else:
        print("\nNo suitable categorical/group field found for grouping in this record set.")
else:
    print("No suitable numeric field detected for EDA.")

## 5. Visualization
Visualize the data: e.g., show the distribution of the selected numeric field, and any relationship to a group (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[numeric_field], kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of '{numeric_field}'")

    if group_field is not None and group_field in df.columns:
        sns.boxplot(x=group_field, y=numeric_field, data=df, ax=axes[1])
        axes[1].set_title(f"'{numeric_field}' by '{group_field}'")
    else:
        df[numeric_field].plot.box(ax=axes[1])
        axes[1].set_title(f"Boxplot of '{numeric_field}'")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field selected for visualization.")

## 6. Conclusion
We have loaded a Croissant data package with `mlcroissant`, explored its structure, and performed basic filtering and normalization on key quantitative fields. Further analysis and targeted biological insights (e.g., protein identification, differential abundance, or peptide sequence analysis) can build on this foundation.